In [2]:
import pandas as pd
import networkx as nx

## Reading in Trade Data

In [3]:
df_2021_2024 = pd.read_csv('data/g50_trade_2021_2024.csv', encoding='latin-1')

In [4]:
df_2011_2020 = pd.read_csv('data/g50_trade_2011_2020.csv', encoding='latin-1')

In [5]:
df_2001_2010 = pd.read_csv('data/g50_trade_2001_2010.csv', encoding='latin-1')

## Select Top X In/Out Neighbours

In [6]:
num_top_in_neighbours = 3 # change number to preference
num_top_out_neighbours = 3 # change number to preference

## Selecting Year of Interest (input required)

In [6]:
# select correct dataframe and year below
df = df_2021_2024[df_2021_2024['refPeriodId'] == 2021]
# df = df_2011_2020[df_2011_2020['refPeriodId'] == 2015]
# df = df_2001_2010[df_2001_2010['refPeriodId'] == 2008]

## Computing Edgelist

In [8]:
graph = nx.Graph()

for year in range(2021,2025):
    df = df_2021_2024[df_2021_2024['refPeriodId'] == year]
    for country in df.reporterISO.unique():
        top_export_rows = df[df['reporterISO']==country].nlargest(num_top_out_neighbours, 'fobvalue')
        for row in top_export_rows.itertuples():
            # graph.add_edge(row.reporterISO, row.partnerISO, weight=row.fobvalue)
            graph.add_edge(row.reporterISO, row.partnerISO, year=year)
    
        top_import_rows = df[df['partnerISO']==country].nlargest(num_top_in_neighbours, 'fobvalue')
        for row in top_import_rows.itertuples():
            # graph.add_edge(row.reporterISO, row.partnerISO, weight=row.fobvalue)
            graph.add_edge(row.reporterISO, row.partnerISO, year=year)

for year in range(2015,2021):
    df = df_2011_2020[df_2011_2020['refPeriodId'] == year]
    for country in df.reporterISO.unique():
        top_export_rows = df[df['reporterISO']==country].nlargest(num_top_out_neighbours, 'fobvalue')
        for row in top_export_rows.itertuples():
            # graph.add_edge(row.reporterISO, row.partnerISO, weight=row.fobvalue)
            graph.add_edge(row.reporterISO, row.partnerISO, year=year)
    
        top_import_rows = df[df['partnerISO']==country].nlargest(num_top_in_neighbours, 'fobvalue')
        for row in top_import_rows.itertuples():
            # graph.add_edge(row.reporterISO, row.partnerISO, weight=row.fobvalue)
            graph.add_edge(row.reporterISO, row.partnerISO, year=year)

## Saving Edgelist

In [8]:
nx.write_edgelist(graph, "2021_33edgelist.csv", data=["weight"], delimiter=',')

## Getting competition network

In [9]:
competition_df = pd.read_csv('data/Competition_graph.csv')

competition_graph = nx.from_pandas_edgelist(competition_df, edge_attr=['year'])

In [10]:
EU = [
    'Austria',
    'Belgium',
    'Bulgaria',
    'Croatia',
    'Cyprus',
    'Czech Republic',
    'Denmark',
    'Estonia',
    'Finland',
    'France',
    'Germany',
    'Greece',
    'Hungary',
    'Ireland',
    'Italy',
    'Latvia',
    'Lithuania',
    'Luxembourg',
    'Malta',
    'Netherlands',
    'Poland',
    'Portugal',
    'Romania',
    'Slovakia',
    'Slovenia',
    'Spain',
    'Sweden'
]

G7 = ['Canada','USA','Japan','United Kingdom'] + EU

In [11]:
competition_graph_10 = nx.Graph()
for source, target, year in competition_graph.edges.data('year'):
    if year >= 2015:
        competition_graph_10.add_edge(source, target)
        if source == 'EU':
            for country in EU:
                competition_graph_10.add_edge(country, target)
        if target == 'EU':
            for country in EU:
                competition_graph_10.add_edge(source, country)
        if source == 'G7':
            for country in G7:
                competition_graph_10.add_edge(country, target)
        if target == 'G7':
            for country in G7:
                competition_graph_10.add_edge(source, country)

In [63]:
competition_graph_10.edges(nbunch='UN')

EdgeDataView([('UN', 'USA'), ('UN', 'Switzerland'), ('UN', 'Canada'), ('UN', 'Australia'), ('UN', 'EU'), ('UN', 'Austria'), ('UN', 'Belgium'), ('UN', 'Bulgaria'), ('UN', 'Croatia'), ('UN', 'Cyprus'), ('UN', 'Czech Republic'), ('UN', 'Denmark'), ('UN', 'Estonia'), ('UN', 'Finland'), ('UN', 'France'), ('UN', 'Germany'), ('UN', 'Greece'), ('UN', 'Hungary'), ('UN', 'Ireland'), ('UN', 'Italy'), ('UN', 'Latvia'), ('UN', 'Lithuania'), ('UN', 'Luxembourg'), ('UN', 'Malta'), ('UN', 'Netherlands'), ('UN', 'Poland'), ('UN', 'Portugal'), ('UN', 'Romania'), ('UN', 'Slovakia'), ('UN', 'Slovenia'), ('UN', 'Spain'), ('UN', 'Sweden'), ('UN', 'Djibouti')])

In [46]:
sorted(competition_graph_10.nodes)

['African Union',
 'Albania',
 'Arab Rep.',
 'Australia',
 'Austria',
 'Bahrain',
 'Belgium',
 'Belize',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Bosnia and Herzegovina,Iceland',
 'Burkina Faso',
 'Canada',
 'Chad',
 'Chile',
 'China',
 'Colombia',
 'Comoros',
 'Czech Republic',
 'Denmark',
 'Djibouti',
 'ECOWAS',
 'EU',
 'Egypt',
 'Estonia',
 'Finland',
 'France',
 'G7',
 'Georgia',
 'Germany',
 'Honduras',
 'Hong Kong',
 'Iceland',
 'Iran',
 'Italy',
 'Japan',
 'Jordan',
 'Kazakhstan',
 'Korea',
 'Kosovo',
 'Latvia',
 'Libya',
 'Liechtenstein',
 'Lithuania',
 'Macedonia',
 'Maldives',
 'Mauritania',
 'Moldova',
 'Monaco',
 'Montenegro',
 'Netherlands',
 'New Zealand',
 'Niger',
 'North',
 'Norway',
 'Peru',
 'Poland',
 'Rep. of Korea',
 'Russia',
 'Saudi Arabia',
 'Senegal',
 'Serbia',
 'Singapore',
 'South',
 'South Africa',
 'Sweden',
 'Switzerland',
 'Taiwan',
 'Türkiye',
 'UN',
 'USA',
 'Ukraine',
 'United Arab Emirates',
 'United Kingdom',
 'Yemen']

In [12]:
node_relabels = {
    'United States': 'USA',
    'Turkey': 'Türkiye',
    'South Korea': 'Rep. of Korea',
    'Russia': 'Russian Federation',
    'Czech Republic' :'Czechia'
    
}
competition_graph = nx.relabel_nodes(competition_graph, node_relabels)

In [21]:
len(list(competition_graph_10.nodes))

89

## Count Triangles in Trading graph - are they triangles in Competition?

In [ ]:
for 